In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/109.0.0.0 Safari/537.36',
'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.9',
'Accept-Language': 'en-US,en;q=0.9',
'Accept-Encoding': 'gzip, deflate, br'}

def clean(df : pd.DataFrame)->pd.DataFrame:

    df["id"] = df["h-property-id"].str.split("ID: ", expand=True)[1]
    df["land-area"] = pd.to_numeric(
    df["h-land-area"].str.split(expand=True)[0].str.replace(",", "."),errors='coerce') * 100

    
    def IDR_EUR_currency()->float:
        source_currency = "IDR"
        target_currency = "EUR"
        response = requests.get(f"https://www.x-rates.com/calculator/?from={source_currency}&to={target_currency}&amount=1")
        soup = BeautifulSoup(response.text, "lxml")
        
        text1 = soup.find(class_="ccOutputCode").previous_sibling
        #text2 = soup.find(class_="ccOutputCode").get_text(strip=True)
        rate = "{}".format(text1)
        return float(rate)

    df["price"] = pd.to_numeric(df["item-price"].str.replace(",", "", regex=False) \
                                .str.replace("IDR", "", regex=False) \
                                .str.replace(".", "", regex=False) \
                                .str.strip().replace("", None, regex=False) \
                                ,errors='coerce') * IDR_EUR_currency()
    df["room"] = df["h-beds"].str.replace("BEDS:", "").astype(float)
    df["area"] = df["h-area"].str.replace("M2", "").str.replace("m²","").str.replace("m2","").str.replace(",","").astype(float)
    df["price_per_m2"] = df["price"] / df["area"]
    df["label-status"] = df["label-status"].str.strip()
    df["hz-label"] = df["hz-label"].str.strip()
    df["source"] = "propertia.com"
    df["query_date"] = pd.Timestamp.today().date().strftime("%Y-%m-%d")
    columns_drop = ["h-property-id", "h-land-area", "item-price" ,  "h-beds" , "h-area"]
    df.drop(columns=columns_drop, inplace=True)

    return df

data = []

# Fetch the webpage content
for page in range(1,25):
        url = f'https://propertia.com/search-results-3/page/{str(page)}/?keyword&bedrooms&min-price&max-price&min-land-area&max-land-area&min-area&label%5B0%5D'
        response = requests.get(url, headers=headers)

        # Parse the HTML content
        soup = BeautifulSoup(response.text, 'lxml')
        listings = soup.findAll("div", {"class" : "item-listing-wrap"})
        for listing in listings:
            temp = {}
            title = listing.find('h2').get_text(strip=True)
            image = listing.find('img')['src'] if listing.find('img') else None
            parent_tag = listing.find('img').parent if listing.find('img') else None
            url = parent_tag.get("href")
            address = listing.find('address').get_text(strip=True)
            details = listing.find('ul', {'class' : 'item-amenities'})
            for detail in details.findAll("li"):
                temp.update({detail.get("class")[0] : detail.text})
            for label in listing.find("div", {"class" : "labels-wrap"}).findAll("a"):
                temp.update({label.get("class")[0] : label.text.capitalize()})

            temp.update({
                'url': url,
                'title': title,
                'address': address,
                'image': image,
                'source' : 'propertia.com'
            })

            data.append(temp)

df = pd.DataFrame(data)
df = clean(df)
df

,h-type,h-custom-price,label-status,hz-label,url,title,address,image,source,id,land-area,price,room,area,price_per_m2,query_date
0,Leasehold Villa,,for sale villa,new listing,https://propertia.com/property/stylish-1-bedro...,Stylish 1-Bedroom Villa in Bali’s Hottest Neig...,Pererenan,https://propertia.com/wp-content/uploads/2025/...,propertia.com,PPV4578,73.0,163500.0,1.0,76.0,2151.315789,2026-04-26
1,Leasehold Villa,,for sale villa,new listing,https://propertia.com/property/luxury-villa-in...,Exceptional Villa Investment in Tropical Perer...,Pererenan,https://propertia.com/wp-content/uploads/2025/...,propertia.com,PPV4577,100.0,206750.0,2.0,94.0,2199.468085,2026-04-26
2,Freehold Land,,for sale land,rarely offered,https://propertia.com/property/freehold-land-i...,FREEHOLD LAND IN A STRATEGIC ULUWATU LOCATION ...,Uluwatu,https://propertia.com/wp-content/uploads/2026/...,propertia.com,PPL3118,1400.0,420000.0,NaN,NaN,NaN,2026-04-26
3,Leasehold Villa,,for sale villa,new listing,https://propertia.com/property/modern-tropical...,MODERN TROPICAL FAMILY HOME IN THE HEART OF CA...,Canggu,https://propertia.com/wp-content/uploads/2026/...,propertia.com,PPV4858,120.0,265000.0,3.0,126.0,2103.174603,2026-04-26
4,Leasehold Villa,,for sale villa,new listing,https://propertia.com/property/turnkey-balanga...,CLEAN MODERN TURNKEY VILLA NEAR BALANGAN BEACH,Balangan,https://propertia.com/wp-content/uploads/2026/...,propertia.com,PPV4857,100.0,158400.0,2.0,120.0,1320.000000,2026-04-26
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
955,Freehold Villa,,for sale villa,great value,https://propertia.com/property/perfect-located...,PERFECT LOCATED VILLA FOR DAILY RENTAL,Berawa,https://propertia.com/wp-content/uploads/2024/...,propertia.com,PPV2228,685.0,675000.0,5.0,272.0,2481.617647,2026-04-26
956,Leasehold Villa,,for sale villa,rarely offered,https://propertia.com/property/outstanding-ico...,OUTSTANDING ICONIC VILLA,Pererenan,https://propertia.com/wp-content/uploads/2024/...,propertia.com,PPV2221,1550.0,1400000.0,7.0,1125.0,1244.444444,2026-04-26
957,Leasehold Villa,,for sale villa,NaN,https://propertia.com/property/huge-beach-fron...,HUGE BEACH FRONT VILLA IN BALIAN,Tabanan,https://propertia.com/wp-content/uploads/2024/...,propertia.com,PPV2211,2000.0,527500.0,4.0,700.0,753.571429,2026-04-26
958,Freehold Villa,,for sale villa,NaN,https://propertia.com/property/majestic-beachf...,MAJESTIC BEACHFRONT ESTATE,Tabanan,https://propertia.com/wp-content/uploads/2024/...,propertia.com,PPV2174,17000.0,2110500.0,4.0,550.0,3837.272727,2026-04-26


In [14]:
df.to_csv("bali_houses.csv", index=False, encoding="utf-8-sig", sep=";")

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/109.0.0.0 Safari/537.36',
'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.9',
'Accept-Language': 'en-US,en;q=0.9',
'Accept-Encoding': 'gzip, deflate, br'}

data = []

for page in range(1,600):

    # Define the URL of the Spitogatos search page
    url = f'https://www.spitogatos.gr/en/for_sale-homes/map-search/order_datemodified_desc/' + \
        f'page_{str(page)}?latitudeLow=34.849875&latitudeHigh=41.319076&longitudeLow=19.500732&longitudeHigh=30.52002&zoom=7'

    # Send a GET request to the URL
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, 'html.parser')
    # Find all listings
    listings = soup.find_all('article', class_='ordered-element')

    for listing in listings:
        title = listing.find('h3', class_='tile__title').get_text(strip=True)
        price = listing.find('p', class_='price__text').get_text(strip=True)
        location = listing.find('h3', class_='tile__location').get_text(strip=True)
        details = listing.find('ul', class_='tile__info').get_text(strip=True)
        floor = listing.find('li', {'title' : 'Floor'}).get_text(strip=True) if listing.find('li', {'title' : 'Floor'}) else None
        bedroom = listing.find('li', {'title' : 'Bedrooms'}).get_text(strip=True) if listing.find('li', {'title' : 'Bedrooms'}) else None
        bathroom = listing.find('li', {'title' : 'Bathrooms'}).get_text(strip=True) if listing.find('li', {'title' : 'Bathrooms'}) else None
        url = listing.find('a')['href']
        image = listing.find('img')['src'] if listing.find('img') else None

        data.append({
            'title': title,
            'price': price,
            'location': location,
            'details': details,
            'floor' : floor,
            'bedroom' : bedroom,
            'bathroom' : bathroom,
            'url': 'https://www.spitogatos.gr' + url,
            'image': image
        })

# Create a DataFrame from the data
df = pd.DataFrame(data)
df

,title,price,location,details,floor,bedroom,bathroom,url,image
0,"Apartment, 125m²","€305,000",Peraia (Thermaikos),3rd3br1ba,3rd,3br,1ba,https://www.spitogatos.gr/en/property/1116380118,https://m2.spitogatos.gr/295422343_300x220.jpg...
1,"Apartment, 73m²","€160,000",Nea Kipseli (Athens - Center),1st2br1ba,1st,2br,1ba,https://www.spitogatos.gr/en/property/1115502059,https://m3.spitogatos.gr/267691580_100x50.jpg
2,"Apartment, 83m²","€320,000",Girokomeio (Athens - Center),4th2br2ba,4th,2br,2ba,https://www.spitogatos.gr/en/property/1114884982,https://m1.spitogatos.gr/253074357_100x50.jpg
3,"Apartment, 114m²","€365,000",Chaniotis (Pallini),1st4br2ba,1st,4br,2ba,https://www.spitogatos.gr/en/property/1116346696,https://m2.spitogatos.gr/242206528_100x50.jpg
4,"Apartment, 130m²","€140,000",Ipsila Alonia (Patra),4th3br1ba,4th,3br,1ba,https://www.spitogatos.gr/en/property/1116405312,https://m3.spitogatos.gr/201633440_100x50.jpg
...,...,...,...,...,...,...,...,...,...
145,"Studio, 42m²","€68,000",Vardaris (Vardaris - Lahanokipi),6th1br1ba,6th,1br,1ba,https://www.spitogatos.gr/en/property/1115006743,https://m2.spitogatos.gr/139375282_100x50.jpg
146,"Apartment, 102m²","€175,000",Omonoia (Athens - Center),6th3br1ba,6th,3br,1ba,https://www.spitogatos.gr/en/property/1116298385,https://m1.spitogatos.gr/253074357_100x50.jpg
147,"Apartment, 85m²","€240,000",Agia Triada (Faliro - Ippokratio),6th2br1ba,6th,2br,1ba,https://www.spitogatos.gr/en/property/1116536145,https://m1.spitogatos.gr/180016500_100x50.jpg
148,"Apartment, 90m²","€164,000",Charokopou (Kallithea),2nd2br1ba,2nd,2br,1ba,https://www.spitogatos.gr/en/property/1116351822,https://m3.spitogatos.gr/207786062_100x50.jpg
